# KEP simulation — 6 loci (B, DR, DQ)

Mirror of `10loci_simulation_v2.ipynb` adapted to the 6-loci experiment:

- Loci: B, DR, DQ (3 loci × 2 alleles = max HLA score 6).
- Resolutions: antigen and allele only (no eplet) → 2 scenarios, 2 result tables.


In [ ]:
import os
import pandas as pd
import numpy as np
import networkx as nx
import warnings
import time
from pathlib import Path
from gurobipy import Model, GRB, quicksum
import scipy.stats as st
warnings.filterwarnings('ignore')

In [ ]:
BASE = Path(os.environ.get("KEP_DATA_DIR", "../../data"))
POOL_DIR = BASE / 'pool_simulations'
MATRICES_DIR = BASE / 'pool_matrices'
RESULTS_DIR = BASE / 'ABO+DSA+HLA' / 'simulation_results'
RESULTS_DIR.mkdir(exist_ok=True)

N_SIMS = 100
LOCI = ['B', 'DR', 'DQ']
ETHCATS = [1, 2, 4, 5, 6, 7]  # OPTN codes

df_pat = pd.read_csv(BASE / 'df_receptores_imputados_final.csv', low_memory=False)
df_pat['WL_ID_CODE'] = df_pat['WL_ID_CODE'].astype('int64')

df_pat['ETHCAT'] = pd.to_numeric(df_pat['ETHCAT'], errors='coerce')

# ===== Simulation parameters =====
SIM_PARAMS = {
    'TOTAL_TIME':       10 * 12,                  
    'ARRIVAL_RATE':     1000 / (10 * 12),
    'MEAN_PATIENCE':    65.1552,   
    'MATCH_RUN':        3,
    'WARMUP_MONTHS':    60,   # months 0-60 = warm-up (not reported)
    'MAX_CYCLE_LENGTH': 3,
    'SEED_BASE':        42,
    'P':                1100,

    # ----- k_graph: HLA threshold for an arc to enter the compatibility graph -----
    'k_graph': {
        'antigen': 2,
        'allele':  1,
    },

    # ----- k_opt[graph_res][opt_res]: HLA threshold for cycle validity -----
    'k_opt': {
        'antigen': {'antigen': 2, 'allele': 1},
        'allele':  {'antigen': 1, 'allele': 1},
    },

    # ----- Z -----
    'Z': {
        'antigen': 6,
        'allele':  6,
    },
}

RESOLUTIONS = ('antigen', 'allele', 'eplet')

COMBOS = [(g, o) for g in RESOLUTIONS for o in RESOLUTIONS]


In [ ]:
# Loads the pool dataframe + the mismatch matrices for one simulation.

def load_sim_data(sim_id):
    pool_df = pd.read_parquet(POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet')
    sim_dir = MATRICES_DIR / f'sim_{sim_id:03d}'

    compat = pd.read_parquet(sim_dir / 'compatibility.parquet').values.astype(np.int8)

    antigen_mm = {L: pd.read_parquet(sim_dir / f'mismatch_antigen_{L}.parquet').values
                  for L in LOCI}
    allele_mm  = {L: pd.read_parquet(sim_dir / f'mismatch_allele_{L}.parquet').values
                  for L in LOCI}

    return {
        'pool_df':    pool_df,
        'compat':     compat,
        'antigen_mm': antigen_mm,
        'allele_mm':  allele_mm,
    }



In [ ]:
# For each resolution, weight = max_score - sum_of_mismatches

MAX_ANTIGEN_6LOCI = 6  
MAX_ALLELE_6LOCI  = 6

def build_weights_6loci(sim_data):
    am = sim_data['antigen_mm']
    al = sim_data['allele_mm']

    sum_antigen = sum(am[L] for L in LOCI).astype(np.int32)
    sum_allele  = sum(al[L] for L in LOCI).astype(np.int32)

    weight_antigen = (MAX_ANTIGEN_6LOCI - sum_antigen).astype(np.int32)
    weight_allele  = (MAX_ALLELE_6LOCI  - sum_allele).astype(np.int32)

    score_B  = (2 - am['B']).astype(np.int32)
    score_DR = (2 - am['DR']).astype(np.int32)
    score_DQ = (2 - am['DQ']).astype(np.int32)

    return {
        'antigen':  weight_antigen,
        'allele':   weight_allele,
        'score_B':  score_B,
        'score_DR': score_DR,
        'score_DQ': score_DQ,
    }



In [ ]:
# BUILD INITIAL GRAPH 

def create_graph(waiting_indices, compat, weight_for_graph, k_graph):
    G = nx.DiGraph()
    G.add_nodes_from(waiting_indices)
    for i in waiting_indices:
        for j in waiting_indices:
            if i == j:
                continue
            if compat[i, j] == 1 and weight_for_graph[i, j] >= k_graph:
                G.add_edge(j, i)
    return G



In [ ]:
# SET EDGE WEIGHTS USING THE OPT RESOLUTION MATRIX
def changing_resolution_weights(G, weight_matrix):
    for u, v in G.edges():
        G[u][v]['weight'] = int(weight_matrix[v, u])

In [ ]:
# KEP CYCLE OPTIMIZATION

def optimization(G, l=3, k_quality=3, Z=6, P=1100):
    total_cycles = list(nx.simple_cycles(G, length_bound=l))
    valid_cycles = [c for c in total_cycles
                    if all(G[u][v]['weight'] >= k_quality
                           for u, v in zip(c, c[1:] + c[:1]))]

    G_opt = nx.DiGraph()
    if not valid_cycles:
        return G_opt, []

    m = Model("kep_optimization")
    m.setParam('OutputFlag', 0)

    x = {tuple(c): m.addVar(vtype=GRB.BINARY, name=f"x_{'_'.join(map(str, c))}")
         for c in valid_cycles}

    m.setObjective(
        quicksum(
            x[tuple(c)] * (
                (len(c) + (1.0 / P) * sum(G[u][v]['weight'] / Z
                                          for u, v in zip(c, c[1:] + c[:1])))
                / P
            )
            for c in valid_cycles
        ),
        GRB.MAXIMIZE
    )

    for node in G.nodes():
        m.addConstr(quicksum(x[tuple(c)] for c in valid_cycles if node in c) <= 1)

    m.optimize()

    selected = []
    if m.status == GRB.OPTIMAL:
        for c in valid_cycles:
            if x[tuple(c)].X > 0.5:
                selected.append(c)
                for i in range(len(c)):
                    u, v = c[i], c[(i + 1) % len(c)]
                    G_opt.add_edge(u, v, weight=G[u][v]['weight'])
    return G_opt, selected

In [ ]:
# RUN ONE SIMULATION FOR A GIVEN (graph_resolution, opt_resolution)
# SET B DESIGN (2x2 matrix)

def run_simulation(sim_id, graph_resolution, opt_resolution,
                   sim_data, weights, params, df_pat):
    pool_df = sim_data['pool_df']
    compat  = sim_data['compat']
    n = len(pool_df)

    weight_for_graph = weights[graph_resolution]
    weight_for_obj   = weights[opt_resolution]
    k_graph = params['k_graph'][graph_resolution]
    k_opt   = params['k_opt'][graph_resolution][opt_resolution]   
    Z       = params['Z'][opt_resolution]

    eth_join = pool_df.merge(df_pat[['WL_ID_CODE', 'ETHCAT']], on='WL_ID_CODE', how='left')
    pair_ethcat = pd.to_numeric(eth_join['ETHCAT'], errors='coerce').fillna(-1).astype(int).values

    ss = np.random.SeedSequence(params['SEED_BASE'] + sim_id * 1000)
    rng_arr, rng_dep = (np.random.default_rng(s) for s in ss.spawn(2))

    available = set(range(n))
    waiting = []
    arrival_t = {}
    departure_t = {}
    historial_cycles = []
    historial_departures = []
    pool_sizes = []
    deadline = {}
    runs_participated = {}
    pool_sizes_by_eth = {e: [] for e in ETHCATS}

    arrivals_by_eth   = {e: 0 for e in ETHCATS}
    departures_by_eth = {e: 0 for e in ETHCATS}

    quality = {(res, e): [] for res in ('antigen', 'allele') for e in ETHCATS}
    quality.update({(loc, e): [] for loc in ('B', 'DR', 'DQ') for e in ETHCATS})

    WARMUP = params.get('WARMUP_MONTHS', 0)
    for month in range(params['TOTAL_TIME']):
        counting = month >= WARMUP
        n_arr = rng_arr.poisson(params['ARRIVAL_RATE'])
        if n_arr > 0:
            new_pairs = rng_arr.choice(list(available), size=n_arr, replace=False)
            for p in new_pairs:
                p_int = int(p)
                arrival_t[p_int] = month
                available.discard(p_int)
                waiting.append(p_int)
                deadline[p_int] = month + rng_dep.exponential(params['MEAN_PATIENCE'])
                e = pair_ethcat[p_int]
                if counting and e in arrivals_by_eth:
                    arrivals_by_eth[e] += 1

        if (month + 1) % params['MATCH_RUN'] == 0 and len(waiting) >= 2:
            if counting: pool_sizes.append(len(waiting))
            for e_ps in (ETHCATS if counting else []):
                pool_sizes_by_eth[e_ps].append(sum(1 for w in waiting if int(pair_ethcat[w]) == e_ps))
            for w in waiting:
                runs_participated[w] = runs_participated.get(w, 0) + 1
            G = create_graph(waiting, compat, weight_for_graph, k_graph)
            changing_resolution_weights(G, weight_for_obj)
            G_opt, selected = optimization(G, l=params['MAX_CYCLE_LENGTH'],
                                              k_quality=k_opt, Z=Z, P=params['P'])

            for u, v in (G_opt.edges() if counting else []):
                e = pair_ethcat[v]
                if e not in arrivals_by_eth:
                    continue
                quality[('antigen', e)].append(int(weights['antigen'][v, u]))
                quality[('allele',  e)].append(int(weights['allele'][v, u]))
                quality[('B',  e)].append(int(weights['score_B'][v, u]))
                quality[('DR', e)].append(int(weights['score_DR'][v, u]))
                quality[('DQ', e)].append(int(weights['score_DQ'][v, u]))

            historial_cycles.extend(selected if counting else [])
            cycled_nodes = {p for c in selected for p in c}
            waiting = [w for w in waiting if w not in cycled_nodes]
            for p_int in cycled_nodes:
                departure_t[int(p_int)] = month

        departed_now = [w for w in waiting if deadline[w] <= month]
        if departed_now:
            ds = set(departed_now)
            waiting = [w for w in waiting if w not in ds]
            for p_int in (departed_now if counting else []):
                historial_departures.append(p_int)
                e = pair_ethcat[p_int]
                if e in departures_by_eth:
                    departures_by_eth[e] += 1

    waiting_times_by_eth = {e: [] for e in ETHCATS}
    cycled_set = {p for c in historial_cycles for p in c}
    for p in cycled_set:
        if p in runs_participated:
            wt = runs_participated[p]
            e = pair_ethcat[p]
            if e in waiting_times_by_eth:
                waiting_times_by_eth[e].append(wt)

    n_total_arr = sum(arrivals_by_eth.values())
    n_total_tx  = sum(len(c) for c in historial_cycles)
    F_total = n_total_tx / max(n_total_arr, 1)
    L_total = len(historial_departures) / max(n_total_arr, 1)
    F_per_eth = {e: sum(1 for c in historial_cycles for p in c if pair_ethcat[p] == e)
                       / max(arrivals_by_eth.get(e, 0), 1)
                 for e in ETHCATS}
    L_per_eth = {e: departures_by_eth[e] / max(arrivals_by_eth.get(e, 0), 1)
                 for e in ETHCATS}

    return {
        'sim_id':           sim_id,
        'graph_resolution': graph_resolution,
        'opt_resolution':   opt_resolution,
        'k_graph':          k_graph,
        'k_opt':            k_opt,
        'Z':                Z,
        'total_arrivals':    n_total_arr,
        'total_transplants': n_total_tx,
        'total_departures':  len(historial_departures),
        'arrivals_by_eth':   arrivals_by_eth,
        'departures_by_eth': departures_by_eth,
        'F_per_eth':         F_per_eth,
        'L_per_eth':         L_per_eth,
        'F_total':           F_total,
        'L_total':           L_total,
        'quality':           quality,
        'waiting_times_by_eth': waiting_times_by_eth,
        'historial_cycles':  historial_cycles,
        'avg_pool_size':     float(np.mean(pool_sizes)) if pool_sizes else 0.0,
        'avg_pool_size_by_eth': {e: float(np.mean(pool_sizes_by_eth[e])) if pool_sizes_by_eth[e] else 0.0 for e in ETHCATS},
    }



In [ ]:

all_results = {combo: [] for combo in COMBOS}
t0 = time.time()

for sim_id in range(N_SIMS):
    sim_data = load_sim_data(sim_id)
    weights  = build_weights_6loci(sim_data)

    for graph_res, opt_res in COMBOS:
        result = run_simulation(sim_id, graph_res, opt_res,
                                sim_data, weights, SIM_PARAMS, df_pat)
        all_results[(graph_res, opt_res)].append(result)

    if (sim_id + 1) % 5 == 0 or sim_id == 0:
        elapsed = (time.time() - t0) / 60
        eta = elapsed / (sim_id + 1) * (N_SIMS - sim_id - 1)
        print(f'  Sim {sim_id+1:3d}/{N_SIMS}')

print(f'\nDone. Total time: {(time.time()-t0)/60:.1f} min')



In [ ]:
# BUILD ONE RESULTS TABLE PER RESOLUTION

def mean_ci(values, ddof=1, conf=0.95):
    arr = np.asarray([v for v in values if pd.notna(v)], dtype=float)
    if len(arr) < 2:
        if len(arr) == 1:
            return arr[0], f"{arr[0]:.3f} [-; -]"
        return float('nan'), 'nan'
    m = arr.mean()
    s = arr.std(ddof=ddof)
    low, high = st.t.interval(conf, len(arr) - 1, loc=m, scale=s / np.sqrt(len(arr)))
    return m, f"{m:.3f} [{low:.3f}; {high:.3f}]"

def build_results_table(results_for_resolution):
    rs = results_for_resolution
    rows = []

    for e in ETHCATS:
        arr_e = np.mean([r['arrivals_by_eth'][e] for r in rs])
        tx_per_sim = [r['F_per_eth'][e] * r['arrivals_by_eth'][e] for r in rs]
        tx_e = np.mean(tx_per_sim)

        F_vals = [r['F_per_eth'][e] for r in rs]
        L_vals = [r['L_per_eth'][e] for r in rs]
        _, txt_F = mean_ci(F_vals)
        _, txt_L = mean_ci(L_vals)

        ant_vals = [np.mean(r['quality'][('antigen', e)]) for r in rs if r['quality'][('antigen', e)]]
        all_vals = [np.mean(r['quality'][('allele',  e)]) for r in rs if r['quality'][('allele',  e)]]
        _, txt_ant = mean_ci(ant_vals)
        _, txt_all = mean_ci(all_vals)

        B_vals  = [np.mean(r['quality'][('B',  e)]) for r in rs if r['quality'][('B',  e)]]
        dr_vals = [np.mean(r['quality'][('DR', e)]) for r in rs if r['quality'][('DR', e)]]
        dq_vals = [np.mean(r['quality'][('DQ', e)]) for r in rs if r['quality'][('DQ', e)]]
        _, txt_B  = mean_ci(B_vals)
        _, txt_dr = mean_ci(dr_vals)
        _, txt_dq = mean_ci(dq_vals)

        wt_vals_flat = [w for r in rs for w in r['waiting_times_by_eth'][e]]
        wt_mean = np.mean(wt_vals_flat) if wt_vals_flat else float('nan')

        still = round(1 - np.mean(F_vals) - np.mean(L_vals), 3)

        rows.append({
            'Ethnicity(s)':                e,
            'Arrivals':                    round(arr_e, 2),
            'Transplants':                 round(tx_e, 2),
            'F(s) (Matched)':              txt_F,
            'HLA(s) Antigen':              txt_ant,
            'HLA(s) Allele':               txt_all,
            'Waiting Time': mean_ci([np.mean(r['waiting_times_by_eth'][e]) for r in rs if r['waiting_times_by_eth'][e]])[1],
            'Pool Size': mean_ci([r['avg_pool_size_by_eth'][e] for r in rs])[1],
            'L(s) (Left Unmatched)':       txt_L,
            '1-F(s)-L(s) (Still in KEP)':  still,
            'HLA B':                       txt_B,
            'HLA DR':                      txt_dr,
            'HLA DQ':                      txt_dq,
        })

    # Entire population row
    F_total_vals = [r['F_total'] for r in rs]
    L_total_vals = [r['L_total'] for r in rs]
    _, txt_F_tot = mean_ci(F_total_vals)
    _, txt_L_tot = mean_ci(L_total_vals)

    arr_tot = np.mean([r['total_arrivals']    for r in rs])
    tx_tot  = np.mean([r['total_transplants'] for r in rs])

    ant_all_per_sim = []; all_all_per_sim = []
    B_all_per_sim   = []; dr_all_per_sim  = []; dq_all_per_sim  = []
    wt_all_per_sim  = []
    for r in rs:
        ant_pool = [v for e in ETHCATS for v in r['quality'][('antigen', e)]]
        all_pool = [v for e in ETHCATS for v in r['quality'][('allele',  e)]]
        B_pool   = [v for e in ETHCATS for v in r['quality'][('B', e)]]
        dr_pool  = [v for e in ETHCATS for v in r['quality'][('DR', e)]]
        dq_pool  = [v for e in ETHCATS for v in r['quality'][('DQ', e)]]
        wt_pool  = [w for e in ETHCATS for w in r['waiting_times_by_eth'][e]]
        if ant_pool: ant_all_per_sim.append(np.mean(ant_pool))
        if all_pool: all_all_per_sim.append(np.mean(all_pool))
        if B_pool:   B_all_per_sim.append(np.mean(B_pool))
        if dr_pool:  dr_all_per_sim.append(np.mean(dr_pool))
        if dq_pool:  dq_all_per_sim.append(np.mean(dq_pool))
        if wt_pool:  wt_all_per_sim.append(np.mean(wt_pool))

    _, txt_ant_tot = mean_ci(ant_all_per_sim)
    _, txt_all_tot = mean_ci(all_all_per_sim)
    _, txt_B_tot   = mean_ci(B_all_per_sim)
    _, txt_dr_tot  = mean_ci(dr_all_per_sim)
    _, txt_dq_tot  = mean_ci(dq_all_per_sim)

    still_tot = round(1 - np.mean(F_total_vals) - np.mean(L_total_vals), 3)

    rows.append({
        'Ethnicity(s)':                'Entire Population',
        'Arrivals':                    round(arr_tot, 2),
        'Transplants':                 round(tx_tot, 2),
        'F(s) (Matched)':              txt_F_tot,
        'HLA(s) Antigen':              txt_ant_tot,
        'HLA(s) Allele':               txt_all_tot,
        'Waiting Time': mean_ci(wt_all_per_sim)[1],
        'Pool Size': mean_ci([r['avg_pool_size'] for r in rs])[1],
        'L(s) (Left Unmatched)':       txt_L_tot,
        '1-F(s)-L(s) (Still in KEP)':  still_tot,
        'HLA B':                       txt_B_tot,
        'HLA DR':                      txt_dr_tot,
        'HLA DQ':                      txt_dq_tot,
    })

    return pd.DataFrame(rows)

In [ ]:

tables = {}
for graph_res, opt_res in COMBOS:
    print(f"\n========== RESULTS — graph={graph_res}, opt={opt_res} ==========\n")
    tbl = build_results_table(all_results[(graph_res, opt_res)])
    tables[(graph_res, opt_res)] = tbl
    display(tbl)



In [ ]:

out_path = RESULTS_DIR / 'results_ABO_DSA_HLA_6loci_4scenarios.xlsx'
with pd.ExcelWriter(out_path) as writer:
    for (graph_res, opt_res), tbl in tables.items():
        sheet = f'g_{graph_res[:3]}_o_{opt_res[:3]}'  
        tbl.to_excel(writer, sheet_name=sheet, index=False)

print(f'Saved: {out_path}')



In [ ]:
# RANK TESTS PER ETHNICITY 

from scipy.stats import wilcoxon, binomtest

ETH_LABELS = {1: 'Caucasian', 2: 'Afroamerican', 4: 'Latin', 5: 'Asian',
              6: 'AmInd', 7: 'PacIsl'}
TESTED_ETHCATS = [1, 2, 4, 5, 6, 7]
SMALL_ETHCATS = {6, 7}

def _mean_or_nan(lst):
    return float(np.mean(lst)) if len(lst) else float('nan')

def _F_eth(r, e):
    return r['F_per_eth'][e] if r['arrivals_by_eth'].get(e, 0) > 0 else float('nan')

def _L_eth(r, e):
    return r['L_per_eth'][e] if r['arrivals_by_eth'].get(e, 0) > 0 else float('nan')

METRIC_EXTRACTORS = {
    'F(s) (Matched)':        (_F_eth, lambda r: r['F_total']),
    'L(s) (Left Unmatched)': (_L_eth, lambda r: r['L_total']),
    'HLA(s) Antigen':        (lambda r, e: _mean_or_nan(r['quality'][('antigen', e)]),
                              lambda r:    _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('antigen', e2)]])),
    'HLA(s) Allele':         (lambda r, e: _mean_or_nan(r['quality'][('allele', e)]),
                              lambda r:    _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('allele', e2)]])),
    'Waiting Time':          (lambda r, e: _mean_or_nan(r['waiting_times_by_eth'][e]),
                              lambda r:    _mean_or_nan([w for e2 in ETHCATS for w in r['waiting_times_by_eth'][e2]])),
    'HLA B':                 (lambda r, e: _mean_or_nan(r['quality'][('B', e)]),
                              lambda r:    _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('B', e2)]])),
    'HLA DR':                (lambda r, e: _mean_or_nan(r['quality'][('DR', e)]),
                              lambda r:    _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('DR', e2)]])),
    'HLA DQ':                (lambda r, e: _mean_or_nan(r['quality'][('DQ', e)]),
                              lambda r:    _mean_or_nan([v for e2 in ETHCATS for v in r['quality'][('DQ', e2)]])),
}

def _paired_rank_tests(diffs):
    non_zero = diffs[diffs != 0]
    if len(non_zero) > 0:
        try:    _, p_w = wilcoxon(non_zero, alternative='two-sided')
        except ValueError: p_w = float('nan')
    else:
        p_w = float('nan')
    n_pos = int((diffs > 0).sum()); n_neg = int((diffs < 0).sum()); n_tot = n_pos + n_neg
    p_s = binomtest(n_pos, n_tot, 0.5, alternative='two-sided').pvalue if n_tot > 0 else float('nan')
    return {
        'p_wilcoxon': p_w, 'p_sign': p_s,
        'n_pos': n_pos, 'n_neg': n_neg, 'n_used': int(len(diffs)),
        'median_diff': float(np.median(diffs)) if len(diffs) else float('nan'),
    }

def rank_test_metric(results_for_combo, eth_fn, overall_fn, ethcats):
    overall_vals = np.array([overall_fn(r) for r in results_for_combo], dtype=float)
    out = {}
    for e in ethcats:
        eth_vals = np.array([eth_fn(r, e) for r in results_for_combo], dtype=float)
        mask = ~(np.isnan(eth_vals) | np.isnan(overall_vals))
        diffs = eth_vals[mask] - overall_vals[mask]
        out[e] = _paired_rank_tests(diffs)
    return out

rank_results_all = {}
for combo in COMBOS:
    rank_results_all[combo] = {}
    for metric_name, (eth_fn, overall_fn) in METRIC_EXTRACTORS.items():
        rank_results_all[combo][metric_name] = rank_test_metric(
            all_results[combo], eth_fn, overall_fn, TESTED_ETHCATS)

def _flags(r):
    f = ''
    if not np.isnan(r['p_wilcoxon']) and r['p_wilcoxon'] < 0.05: f += 'W'
    if not np.isnan(r['p_sign'])     and r['p_sign']     < 0.05: f += 'S'
    return f

for graph_res, opt_res in COMBOS:
    print(f"\n===== graph={graph_res}, opt={opt_res} — flags [W = Wilcoxon p<.05, S = sign test p<.05] =====")
    header = f"{'Metric':24s}" + "".join(
        f"{(ETH_LABELS[e] + ('*' if e in SMALL_ETHCATS else '')):>14s}" for e in TESTED_ETHCATS)
    print(header)
    print("-" * len(header))
    for m in METRIC_EXTRACTORS:
        row = f"{m:24s}"
        for e in TESTED_ETHCATS:
            row += f"{('[' + _flags(rank_results_all[(graph_res, opt_res)][m][e]) + ']'):>14s}"
        print(row)



In [ ]:

from openpyxl import Workbook
from openpyxl.styles import Font

ALPHA = 0.05
METRIC_COLUMNS = list(METRIC_EXTRACTORS.keys())

wb = Workbook()
wb.remove(wb.active)

for combo in COMBOS:
    graph_res, opt_res = combo
    ws = wb.create_sheet(f'g_{graph_res[:3]}_o_{opt_res[:3]}')
    tbl = tables[combo]

    for col_idx, col_name in enumerate(tbl.columns, start=1):
        c = ws.cell(row=1, column=col_idx, value=col_name)
        c.font = Font(bold=True)

    for row_pos, (_, row) in enumerate(tbl.iterrows(), start=2):
        eth_raw = row['Ethnicity(s)']
        is_eth_code = isinstance(eth_raw, (int, np.integer))
        eth_code = int(eth_raw) if is_eth_code else None

        for col_idx, col_name in enumerate(tbl.columns, start=1):
            val = row[col_name]
            if col_name == 'Ethnicity(s)' and is_eth_code:
                label = ETH_LABELS.get(eth_code, str(eth_raw))
                val = label + ('*' if eth_code in SMALL_ETHCATS else '')
            cell = ws.cell(row=row_pos, column=col_idx, value=val)

            if col_name in METRIC_COLUMNS and eth_code in TESTED_ETHCATS:
                r = rank_results_all[combo][col_name][eth_code]
                bold      = (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA
                underline = (not np.isnan(r['p_sign']))     and r['p_sign']     < ALPHA
                if bold or underline:
                    cell.font = Font(bold=bold, underline='single' if underline else None)

    for col_idx, col_name in enumerate(tbl.columns, start=1):
        col_letter = chr(64 + col_idx) if col_idx <= 26 else 'A' + chr(64 + col_idx - 26)
        ws.column_dimensions[col_letter].width = max(14, len(col_name) + 2)

    foot_row = len(tbl) + 4
    ws.cell(row=foot_row, column=1, value=f'graph={graph_res}, opt={opt_res}  —  significance vs entire population:')
    fb = ws.cell(row=foot_row+1, column=1, value='   bold       = Wilcoxon signed-rank p < 0.05')
    fu = ws.cell(row=foot_row+2, column=1, value='   underlined = sign test p < 0.05')
    fb.font = Font(bold=True)
    fu.font = Font(underline='single')

out_path = RESULTS_DIR / 'results_ABO_DSA_HLA_6loci_4scenarios_significance.xlsx'
wb.save(out_path)
print(f'Saved: {out_path}')

In [ ]:
OUT_SIG_FILENAME = 'results_ABO_DSA_HLA_6loci_4scenarios_significance.xlsx'

# PAIRWISE TESTS BETWEEN DIAGONAL SCENARIOS (antigen/antigen, allele/allele)
# Adds dagger marks (†) to the diagonal sheets of the significance Excel

DIAGONAL_RESOLUTIONS = ('antigen', 'allele')
DIAGONAL_COMBOS = [(r, r) for r in DIAGONAL_RESOLUTIONS]

def _paired_rank_tests_diag(diffs):
    non_zero = diffs[diffs != 0]
    if len(non_zero) > 0:
        try:    _, p_w = wilcoxon(non_zero, alternative='two-sided')
        except ValueError: p_w = float('nan')
    else:
        p_w = float('nan')
    n_pos = int((diffs > 0).sum()); n_neg = int((diffs < 0).sum())
    n_tot = n_pos + n_neg
    p_s = binomtest(n_pos, n_tot, 0.5, alternative='two-sided').pvalue if n_tot > 0 else float('nan')
    return {'p_wilcoxon': p_w, 'p_sign': p_s, 'n_pos': n_pos, 'n_neg': n_neg,
            'n_used': int(len(diffs)),
            'median_diff': float(np.median(diffs)) if len(diffs) else float('nan')}

def rank_test_metric_pair_diag(results_A, results_B, eth_fn, ethcats):
    n = min(len(results_A), len(results_B))
    out = {}
    for e in ethcats:
        vals_A = np.array([eth_fn(results_A[i], e) for i in range(n)], dtype=float)
        vals_B = np.array([eth_fn(results_B[i], e) for i in range(n)], dtype=float)
        mask = ~(np.isnan(vals_A) | np.isnan(vals_B))
        diffs = vals_A[mask] - vals_B[mask]
        out[e] = _paired_rank_tests_diag(diffs)
    return out

# Build pairwise results — diagonal vs diagonal
rank_results_pairwise_diag = {res_A: {} for res_A in DIAGONAL_RESOLUTIONS}
for res_A in DIAGONAL_RESOLUTIONS:
    for res_B in DIAGONAL_RESOLUTIONS:
        if res_A == res_B: continue
        rank_results_pairwise_diag[res_A][res_B] = {}
        for metric_name, (eth_fn, _) in METRIC_EXTRACTORS.items():
            rank_results_pairwise_diag[res_A][res_B][metric_name] = rank_test_metric_pair_diag(
                all_results[(res_A, res_A)], all_results[(res_B, res_B)], eth_fn, TESTED_ETHCATS)

rank_results_pairwise_diag_overall = {res_A: {} for res_A in DIAGONAL_RESOLUTIONS}
for res_A in DIAGONAL_RESOLUTIONS:
    for res_B in DIAGONAL_RESOLUTIONS:
        if res_A == res_B: continue
        rank_results_pairwise_diag_overall[res_A][res_B] = {}
        for metric_name, (_, overall_fn) in METRIC_EXTRACTORS.items():
            A = all_results[(res_A, res_A)]; B = all_results[(res_B, res_B)]
            n = min(len(A), len(B))
            vals_A = np.array([overall_fn(A[i]) for i in range(n)], dtype=float)
            vals_B = np.array([overall_fn(B[i]) for i in range(n)], dtype=float)
            mask = ~(np.isnan(vals_A) | np.isnan(vals_B))
            rank_results_pairwise_diag_overall[res_A][res_B][metric_name] = _paired_rank_tests_diag(vals_A[mask] - vals_B[mask])

print("\n===== PAIRWISE BETWEEN DIAGONAL scenarios (antigen/antigen, allele/allele) =====")
print("    [W = Wilcoxon p<.05, S = sign test p<.05]")
seen = set()
for res_A in DIAGONAL_RESOLUTIONS:
    for res_B in DIAGONAL_RESOLUTIONS:
        if res_A == res_B: continue
        key = tuple(sorted([res_A, res_B]))
        if key in seen: continue
        seen.add(key)
        a, b = key
        print(f"\n--- {a}/{a} vs {b}/{b} ---")
        hdr = f"{'Metric':24s}" + "".join(f"{ETH_LABELS[e]:>14s}" for e in TESTED_ETHCATS)
        print(hdr); print("-" * len(hdr))
        for m in METRIC_EXTRACTORS:
            row = f"{m:24s}"
            for e in TESTED_ETHCATS:
                row += f"{('[' + _flags(rank_results_pairwise_diag[a][b][m][e]) + ']'):>14s}"
            print(row)

# Re-save Excel with pairwise marks on diagonal sheets only
OTHER_DIAG = {r: [x for x in DIAGONAL_RESOLUTIONS if x != r] for r in DIAGONAL_RESOLUTIONS}

def _pairwise_marks_diag(opt_res, metric, eth_code):
    if eth_code not in TESTED_ETHCATS: return ''
    others = OTHER_DIAG[opt_res]
    marks = ''
    for sym, other in zip(['†'], others):
        r = rank_results_pairwise_diag[opt_res][other][metric][eth_code]
        if (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA:
            marks += sym
    return marks

def _pairwise_marks_diag_overall(opt_res, metric):
    others = OTHER_DIAG[opt_res]
    marks = ''
    for sym, other in zip(['†', '‡'], others):
        r = rank_results_pairwise_diag_overall[opt_res][other][metric]
        if (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA:
            marks += sym
    return marks

wb = Workbook()
wb.remove(wb.active)

for combo in COMBOS:
    graph_res, opt_res = combo
    ws = wb.create_sheet(f'g_{graph_res[:3]}_o_{opt_res[:3]}')
    tbl = tables[combo]
    is_diagonal = (graph_res == opt_res)

    for col_idx, col_name in enumerate(tbl.columns, start=1):
        c = ws.cell(row=1, column=col_idx, value=col_name)
        c.font = Font(bold=True)

    for row_pos, (_, row) in enumerate(tbl.iterrows(), start=2):
        eth_raw = row['Ethnicity(s)']
        is_eth_code = isinstance(eth_raw, (int, np.integer))
        eth_code = int(eth_raw) if is_eth_code else None
        for col_idx, col_name in enumerate(tbl.columns, start=1):
            val = row[col_name]
            if col_name == 'Ethnicity(s)' and is_eth_code:
                val = ETH_LABELS.get(eth_code, str(eth_raw)) + ('*' if eth_code in SMALL_ETHCATS else '')
            cell = ws.cell(row=row_pos, column=col_idx, value=val)
            if col_name in METRIC_COLUMNS and eth_code in TESTED_ETHCATS:
                r = rank_results_all[combo][col_name][eth_code]
                bold      = (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA
                underline = (not np.isnan(r['p_sign']))     and r['p_sign']     < ALPHA
                if bold or underline:
                    cell.font = Font(bold=bold, underline='single' if underline else None)
                # Pairwise marks ONLY on diagonal sheets
                if is_diagonal:
                    marks = _pairwise_marks_diag(opt_res, col_name, eth_code)
                    if marks:
                        if isinstance(val, str):
                            cell.value = val + marks
                        elif isinstance(val, (int, float)) and not (isinstance(val, float) and np.isnan(val)):
                            cell.number_format = f'0.000"{marks}"'
            if col_name in METRIC_COLUMNS and (eth_code is None) and is_diagonal:
                marks = _pairwise_marks_diag_overall(opt_res, col_name)
                if marks:
                    if isinstance(val, str):
                        cell.value = val + marks
                    elif isinstance(val, (int, float)) and not (isinstance(val, float) and np.isnan(val)):
                        cell.number_format = f'0.000"{marks}"'

    for col_idx, col_name in enumerate(tbl.columns, start=1):
        col_letter = chr(64 + col_idx) if col_idx <= 26 else 'A' + chr(64 + col_idx - 26)
        ws.column_dimensions[col_letter].width = max(14, len(col_name) + 2)

    foot_row = len(tbl) + 4
    ws.cell(row=foot_row, column=1, value=f'graph={graph_res}, opt={opt_res}  —  significance vs entire population:')
    fb = ws.cell(row=foot_row+1, column=1, value='   bold       = Wilcoxon signed-rank p < 0.05')
    fu = ws.cell(row=foot_row+2, column=1, value='   underlined = sign test p < 0.05')
    fb.font = Font(bold=True)
    fu.font = Font(underline='single')

    if is_diagonal:
        others = OTHER_DIAG[opt_res]
        if len(others) >= 1:
            ws.cell(row=foot_row+3, column=1,
                    value=f'   †  = Wilcoxon p < 0.05 — diagonal ({opt_res}/{opt_res}) vs ({others[0]}/{others[0]})')

out_path = RESULTS_DIR / OUT_SIG_FILENAME
wb.save(out_path)
print(f'\n✓ Saved (with pairwise marks on diagonals): {out_path}')
